# 🎫 AI Support Ticket Triage System

**Week 1 · Project 1** — built with plain LLM APIs (OpenAI, with optional Groq / Gemini).

Businesses receive support tickets from email, chat, WhatsApp, forms, and helpdesk tools. Each ticket must be understood, categorized, prioritized, routed, summarized, and answered. Manual triage is slow and inconsistent. This notebook automates it with a single **structured** LLM call per ticket.

## What the system does (assignment requirements)

| # | Requirement | Where it is handled |
|---|-------------|---------------------|
| 1 | Accept a customer support message | `triage_ticket(message, channel)` |
| 2 | Classify the ticket **category** | `category` field (7 classes) |
| 3 | Detect **priority** level | `priority` field (low/medium/high/urgent) |
| 4 | Assign the correct **department** | `department` field + `enforce_routing()` (intent → team) |
| 5 | Generate a short **issue summary** | `issue_summary` field |
| 6 | Generate a **customer-facing reply** | `customer_reply` field |
| 7 | Decide whether **human escalation** is required | `escalate_to_human` + `apply_safety_net()` |

## Design at a glance

```
customer message
       │
       ▼
  ┌──────────────────────────────┐
  │ ONE structured LLM call       │  Pydantic schema forces valid,
  │ (OpenAI Responses API .parse) │  typed JSON — no fragile parsing
  └──────────────────────────────┘
       │  category, priority, summary, reply, escalate, confidence
       ▼
  enforce_routing()   → department is derived from category (deterministic)
       │
       ▼
  apply_safety_net()  → Python rules FORCE escalation on complaints,
       │                 urgent/high priority, low confidence (<0.6),
       │                 or refund/legal/security keywords
       ▼
  triaged ticket  → table + human-readable report + CSV export
```

**Providers:** OpenAI is the default and uses the native `responses.parse()` structured-output API. Groq and Gemini are supported through their OpenAI-compatible endpoints (JSON mode + Pydantic validation), so the whole system runs through a single SDK.


## 1. Install & configure

Add your key(s) to **Colab → Secrets** (🔑 icon, left sidebar) with names `OPENAI_API_KEY` (required), and optionally `GROQ_API_KEY`, `GEMINI_API_KEY`. Enable *Notebook access* for each.

In [ ]:
!pip -q install "openai>=1.50.0" "pydantic>=2.5" pandas

In [ ]:
import os
import json
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from openai import OpenAI

# All three providers speak the OpenAI wire format, so ONE SDK handles everything.
# Change models here in ONE place.
PROVIDERS = {
    "openai": {"model": "gpt-4.1-mini",            "base_url": None,
               "key": "OPENAI_API_KEY"},
    "groq":   {"model": "llama-3.3-70b-versatile", "base_url": "https://api.groq.com/openai/v1",
               "key": "GROQ_API_KEY"},
    "gemini": {"model": "gemini-2.5-flash",        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "key": "GEMINI_API_KEY"},
}
DEFAULT_PROVIDER = "openai"

def get_secret(name):
    """Read a secret from Colab Secrets if available, else from the environment."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

_clients = {}
def get_client(provider):
    """Lazily build and cache an OpenAI-compatible client for a provider."""
    if provider not in _clients:
        cfg = PROVIDERS[provider]
        key = get_secret(cfg["key"])
        if not key:
            raise RuntimeError(f"Missing API key secret '{cfg['key']}' for provider '{provider}'.")
        _clients[provider] = (OpenAI(api_key=key, base_url=cfg["base_url"])
                              if cfg["base_url"] else OpenAI(api_key=key))
    return _clients[provider]

available = [p for p, cfg in PROVIDERS.items() if get_secret(cfg["key"])]
print("Providers with keys available:", available or "NONE - please set OPENAI_API_KEY")

## 2. Business taxonomy & routing rules

The **intent-based routing** idea: the model classifies the *category*, and everything else (which team owns it, whether to escalate) is driven from that. Department routing is deterministic (a fixed map) so it can never drift.

In [ ]:
# Category -> owning team (deterministic routing; overrides whatever the model guesses)
CATEGORY_TO_DEPARTMENT = {
    "billing_and_payments": "Billing",
    "technical_issue":      "Technical Support",
    "account_access":       "Account Management",
    "product_inquiry":      "Sales",
    "complaint":            "Customer Success",
    "feature_request":      "Product",
    "general_query":        "General Support",
}

PRIORITY_ICON = {"low": "🟢", "medium": "🟡", "high": "🟠", "urgent": "🔴"}

# If any of these appear, we force a human review regardless of model output.
SENSITIVE_KEYWORDS = [
    "refund", "chargeback", "lawyer", "legal", "gdpr", "sue", "lawsuit",
    "cancel", "terminate", "data loss", "lost all", "breach", "hacked",
    "fraud", "unauthorized", "compensation",
]

## 3. Structured output schema

The single most important reliability decision: instead of asking the model to "return JSON" and hoping, we declare a **Pydantic schema**. OpenAI's Structured Outputs then *guarantee* the reply matches it — every field present, every enum value valid — so downstream code never crashes on shape.

In [ ]:
class TicketTriageOutput(BaseModel):
    category: Literal[
        "billing_and_payments", "technical_issue", "account_access",
        "product_inquiry", "complaint", "feature_request", "general_query",
    ] = Field(description="Single best category for the ticket.")

    priority: Literal["low", "medium", "high", "urgent"] = Field(
        description="Business priority of the ticket.")

    department: Literal[
        "Billing", "Technical Support", "Account Management",
        "Sales", "Customer Success", "Product", "General Support",
    ] = Field(description="Owning team (will be re-derived from category for safety).")

    issue_summary: str = Field(
        description="One concise, factual sentence describing the core issue, for internal agents.")

    customer_reply: str = Field(
        description="Short (3-5 sentence) empathetic, professional reply addressed to the customer.")

    escalate_to_human: bool = Field(
        description="True if a human agent must review/handle this ticket.")

    escalation_reason: str = Field(
        description="Brief reason for escalation, or 'None' if no escalation is needed.")

    confidence: float = Field(
        description="Confidence in the classification, from 0.0 to 1.0.")

# Peek at the JSON Schema that gets sent to the model:
TicketTriageOutput.model_json_schema()

## 4. System prompt (the triage policy)

This is the "business logic" the model follows: category definitions, priority rules, department mapping, escalation triggers, and reply style.

In [ ]:
SYSTEM_PROMPT = """You are an expert customer-support triage assistant for a B2B SaaS company.
You receive ONE customer support message (with the channel it arrived on) and must return a structured triage decision.

Work in this order (intent-based routing): first decide the CATEGORY, then let it drive priority, department, escalation, and the reply.

CATEGORY - choose the single best fit:
- billing_and_payments : charges, invoices, refunds, subscriptions, pricing, plan changes
- technical_issue      : bugs, errors, crashes, outages, performance, API/integration problems
- account_access       : login, passwords, 2FA, locked/suspended accounts, account security
- product_inquiry      : pre-sales or how-to questions about capabilities, plans, upgrades
- complaint            : clear dissatisfaction, escalations, or threats to cancel/churn
- feature_request      : asking for a new feature or enhancement
- general_query        : anything else that does not fit the above

PRIORITY:
- urgent : production outage, security/fraud, data loss, legal/compliance threat, or the customer is completely blocked on a paid service
- high   : a single customer is blocked, an angry customer, a refund/billing dispute, or a repeated unresolved issue
- medium : a partial problem with a workaround, or a non-blocking bug
- low    : general questions, feature requests, minor or cosmetic issues

DEPARTMENT - route to the owning team:
- Billing            -> billing_and_payments
- Technical Support  -> technical_issue
- Account Management -> account_access
- Sales              -> product_inquiry
- Customer Success   -> complaint
- Product            -> feature_request
- General Support    -> general_query

ESCALATE_TO_HUMAN = true when ANY of these hold:
- priority is urgent or high
- the message involves security, fraud, data loss, or legal/compliance (e.g. GDPR)
- the customer is angry or threatens to cancel/churn
- a refund or monetary dispute is requested
- the request needs human judgement, is ambiguous, or is outside standard policy
Otherwise escalate_to_human = false.

ISSUE_SUMMARY:
- One concise, factual sentence describing the core issue, written for an internal agent.

CUSTOMER_REPLY:
- A short (3-5 sentence) warm, professional reply addressed to the customer.
- Acknowledge the issue with empathy, state the next step, and set expectations.
- If escalating, tell the customer a specialist will follow up.
- Never invent specific facts (order numbers, exact dates, refund amounts) that are not in the message.

ESCALATION_REASON:
- A brief reason if escalating; otherwise the string "None".

CONFIDENCE:
- Your confidence in the classification, a float from 0.0 to 1.0.

Base every decision only on the content of the message. If unsure, lower your confidence and prefer escalating to a human."""

# Extra instruction appended only for JSON-mode providers (Groq / Gemini):
JSON_INSTRUCTION = """Return ONLY a single JSON object (no markdown, no commentary) with EXACTLY these keys:
"category", "priority", "department", "issue_summary", "customer_reply", "escalate_to_human", "escalation_reason", "confidence".
Use only the allowed values defined above. "escalate_to_human" must be a boolean and "confidence" a number between 0 and 1."""

print("System prompt ready (%d chars)." % len(SYSTEM_PROMPT))

## 5. Core triage function (multi-provider) + safety net

- **OpenAI** → native `responses.parse()` returns a pre-validated `TicketTriageOutput`.
- **Groq / Gemini** → JSON mode (`response_format={"type": "json_object"}`), then we validate with Pydantic ourselves.
- `enforce_routing()` fixes the department from the category.
- `apply_safety_net()` applies hard business rules that can **override** the model to force human escalation.

In [ ]:
def _user_input(message, channel):
    return f"Channel: {channel}\n\nCustomer support message:\n---\n{message}\n---"

def _call_openai_native(client, model, message, channel):
    resp = client.responses.parse(
        model=model,
        instructions=SYSTEM_PROMPT,
        input=_user_input(message, channel),
        text_format=TicketTriageOutput,
        temperature=0,           # deterministic, consistent triage
        max_output_tokens=900,   # cost / latency guardrail
    )
    if getattr(resp, "status", None) == "incomplete":
        raise RuntimeError(f"Incomplete response: {getattr(resp, 'incomplete_details', None)}")
    if resp.output_parsed is None:
        raise RuntimeError("No parsed output (possible refusal).")
    return resp.output_parsed

def _call_json_mode(client, model, message, channel):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT + "\n\n" + JSON_INSTRUCTION},
            {"role": "user",   "content": _user_input(message, channel)},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return TicketTriageOutput.model_validate_json(resp.choices[0].message.content)

def enforce_routing(t: TicketTriageOutput) -> TicketTriageOutput:
    """Department is authoritative from the category map, never the raw model guess."""
    t.department = CATEGORY_TO_DEPARTMENT.get(t.category, "General Support")
    return t

def apply_safety_net(t: TicketTriageOutput, message: str) -> TicketTriageOutput:
    """Force human escalation when hard business rules are triggered."""
    reasons = []
    if t.escalate_to_human and t.escalation_reason and t.escalation_reason.strip().lower() != "none":
        reasons.append(t.escalation_reason.strip())
    if t.confidence < 0.6:
        reasons.append(f"low model confidence ({t.confidence:.2f})")
    if t.category == "complaint":
        reasons.append("category is complaint")
    if t.priority in ("urgent", "high"):
        reasons.append(f"{t.priority} priority")
    hits = sorted({k for k in SENSITIVE_KEYWORDS if k in message.lower()})
    if hits:
        reasons.append("sensitive keywords: " + ", ".join(hits))

    if reasons:
        t.escalate_to_human = True
        t.escalation_reason = "; ".join(dict.fromkeys(reasons))  # dedupe, keep order
    else:
        t.escalate_to_human = False
        t.escalation_reason = "None"
    return t

def triage_ticket(message: str, channel: str = "email", provider: str = DEFAULT_PROVIDER) -> TicketTriageOutput:
    """Full pipeline: LLM classify -> enforce routing -> safety-net escalation."""
    client = get_client(provider)
    model = PROVIDERS[provider]["model"]
    if provider == "openai":
        t = _call_openai_native(client, model, message, channel)
    else:
        t = _call_json_mode(client, model, message, channel)
    t = enforce_routing(t)
    t = apply_safety_net(t, message)
    return t

def print_triage(ticket_id, subject, channel, message, t: TicketTriageOutput):
    msg = message.strip()
    print("=" * 74)
    print(f"🎫 {ticket_id}  |  channel: {channel}  |  subject: {subject}")
    print("-" * 74)
    print(f"Message   : {msg[:170]}{'...' if len(msg) > 170 else ''}")
    print(f"Category  : {t.category}")
    print(f"Priority  : {PRIORITY_ICON.get(t.priority, '')} {t.priority.upper()}")
    print(f"Department: {t.department}")
    print(f"Confidence: {t.confidence:.2f}")
    print(f"Escalate  : {'YES -> ' + t.escalation_reason if t.escalate_to_human else 'No'}")
    print(f"Summary   : {t.issue_summary}")
    print("Reply     :")
    print("   " + t.customer_reply.replace("\n", "\n   "))
    print()

## 6. Single-ticket smoke test

Run one message end-to-end to confirm everything is wired up.

In [ ]:
demo_msg = "I was charged twice for my subscription this month. Please refund the duplicate $49 charge as soon as possible."
demo = triage_ticket(demo_msg, channel="email", provider=DEFAULT_PROVIDER)
print_triage("DEMO-1", "Double charge", "email", demo_msg, demo)
demo.model_dump()

## 7. Dataset

The notebook ships its own dataset so it runs end-to-end with no manual upload. The same 16 tickets are also provided as `dataset/support_tickets.csv` in the submission. Tickets span every channel, category, and priority level (including edge cases: security, data loss, GDPR, angry churn risk, and a happy customer).

In [ ]:
tickets = [
    {"ticket_id": "T-1001", "channel": "email",    "customer_name": "Rahul Sharma",  "subject": "Charged twice this month",              "message": "I was billed twice for my Pro subscription this month. I can see two identical charges of $49 on my card statement. Please refund the duplicate charge as soon as possible."},
    {"ticket_id": "T-1002", "channel": "chat",     "customer_name": "Emily Carter",  "subject": "App crashes when uploading files",       "message": "Every time I try to upload a PDF larger than 10MB the app freezes and then crashes. This has been happening since yesterday and I cannot get my work done."},
    {"ticket_id": "T-1003", "channel": "whatsapp", "customer_name": "Mohammed Ali",  "subject": "Cannot log in",                          "message": "I forgot my password and the reset email is not arriving. I have already checked my spam folder. I need to access my account today for a client meeting."},
    {"ticket_id": "T-1004", "channel": "form",     "customer_name": "Priya Nair",    "subject": "Please add dark mode",                   "message": "It would be great if you could add a dark mode theme to the dashboard. My eyes get tired working late at night."},
    {"ticket_id": "T-1005", "channel": "email",    "customer_name": "George Miller", "subject": "Extremely frustrated with downtime",     "message": "This is the third time this month your service has gone down during our business hours. We are losing customers because of this. If it happens again we will cancel our contract and move to a competitor."},
    {"ticket_id": "T-1006", "channel": "helpdesk", "customer_name": "Sara Kim",      "subject": "Production API returning 500 errors",    "message": "Our production integration is receiving HTTP 500 errors from your /v1/orders endpoint for the last 30 minutes. This is blocking all of our checkout flows. Please treat this as urgent."},
    {"ticket_id": "T-1007", "channel": "email",    "customer_name": "Daniel Lee",    "subject": "How do I upgrade my plan?",              "message": "I am currently on the Starter plan and want to move to the Business plan. Can you tell me how to upgrade and whether the price is prorated?"},
    {"ticket_id": "T-1008", "channel": "chat",     "customer_name": "Aisha Khan",    "subject": "Suspicious login on my account",         "message": "I received an email saying there was a login from a device in another country that I do not recognize. I think someone has accessed my account. Please help me secure it immediately."},
    {"ticket_id": "T-1009", "channel": "email",    "customer_name": "Robert Brown",  "subject": "Requesting a refund",                    "message": "I am not satisfied with the product and would like a full refund for the annual plan I purchased last week. It does not meet the needs my sales rep promised."},
    {"ticket_id": "T-1010", "channel": "whatsapp", "customer_name": "Neha Gupta",    "subject": "Need a tax invoice",                     "message": "Could you please send me a proper tax invoice for my last payment? My accounting team needs it for our records."},
    {"ticket_id": "T-1011", "channel": "form",     "customer_name": "Tom Wilson",    "subject": "Slack integration request",              "message": "Do you have any plans to build a native Slack integration? We would love to receive alerts directly in our team channel."},
    {"ticket_id": "T-1012", "channel": "email",    "customer_name": "Linda Davis",   "subject": "Lost all my data after sync",            "message": "After the last sync my entire project workspace is empty. All my documents and boards are gone. This is critical data for my business and I need it recovered urgently."},
    {"ticket_id": "T-1013", "channel": "chat",     "customer_name": "Kevin Patel",   "subject": "Dashboard is very slow",                 "message": "The reporting dashboard takes almost a minute to load and often times out. It used to be fast. Is there anything I can do?"},
    {"ticket_id": "T-1014", "channel": "email",    "customer_name": "Maria Garcia",  "subject": "Want to cancel my subscription",         "message": "Please cancel my subscription at the end of the current billing cycle. I no longer need the service. Confirm once it is done."},
    {"ticket_id": "T-1015", "channel": "helpdesk", "customer_name": "James Nguyen",  "subject": "GDPR data deletion request",             "message": "Under GDPR I am requesting that you delete all personal data associated with my account and provide written confirmation once completed."},
    {"ticket_id": "T-1016", "channel": "email",    "customer_name": "Olivia Martin", "subject": "Thank you and a quick question",         "message": "Just wanted to say your support team was fantastic last week. Also, is there a mobile app available for iOS? Thanks again."},
]

tickets_df = pd.DataFrame(tickets)
tickets_df.to_csv("support_tickets.csv", index=False)   # (re)create the dataset file
print(f"Loaded {len(tickets_df)} tickets")
tickets_df[["ticket_id", "channel", "subject"]]

## 8. Batch-triage every ticket

Run the whole dataset through the pipeline and collect the structured results into a table.

In [ ]:
BATCH_PROVIDER = DEFAULT_PROVIDER   # set to "groq" or "gemini" to try another provider

records = []
for row in tickets_df.itertuples(index=False):
    try:
        t = triage_ticket(row.message, row.channel, provider=BATCH_PROVIDER)
        records.append({
            "ticket_id": row.ticket_id, "channel": row.channel, "subject": row.subject,
            "category": t.category, "priority": t.priority, "department": t.department,
            "confidence": round(t.confidence, 2), "escalate": t.escalate_to_human,
            "escalation_reason": t.escalation_reason,
            "issue_summary": t.issue_summary, "customer_reply": t.customer_reply,
        })
        print(f"✓ {row.ticket_id}: {t.category} / {t.priority} / {t.department}"
              f" / escalate={t.escalate_to_human}")
    except Exception as e:
        print(f"✗ {row.ticket_id} FAILED: {e}")
        records.append({"ticket_id": row.ticket_id, "channel": row.channel,
                        "subject": row.subject, "category": "ERROR",
                        "escalation_reason": str(e)})

results_df = pd.DataFrame(records)
results_df[["ticket_id", "category", "priority", "department", "confidence", "escalate"]]

### Save the results

In [ ]:
results_df.to_csv("triage_results.csv", index=False)
print("Saved -> triage_results.csv")
results_df.head()

## 9. Human-readable triage report

Print each ticket with its full triage decision and the drafted customer reply.

In [ ]:
src_by_id = {r.ticket_id: r for r in tickets_df.itertuples(index=False)}
for rec in records:
    if rec.get("category") == "ERROR":
        print(f"{rec['ticket_id']}: ERROR - {rec.get('escalation_reason','')}")
        continue
    src = src_by_id[rec["ticket_id"]]
    t = TicketTriageOutput(
        category=rec["category"], priority=rec["priority"], department=rec["department"],
        issue_summary=rec["issue_summary"], customer_reply=rec["customer_reply"],
        escalate_to_human=rec["escalate"], escalation_reason=rec["escalation_reason"],
        confidence=rec["confidence"],
    )
    print_triage(rec["ticket_id"], src.subject, src.channel, src.message, t)

## 10. (Bonus) Multi-turn conversation memory per ticket

Real tickets have follow-ups. `conversation_memory` keeps prior turns per `ticket_id` so a follow-up message is triaged *with* its context.

In [ ]:
conversation_memory = {}   # ticket_id -> list of prior turn strings

def triage_conversation(ticket_id, message, channel="email", provider=DEFAULT_PROVIDER):
    history = conversation_memory.setdefault(ticket_id, [])
    context = ""
    if history:
        context = "Earlier in this ticket:\n" + "\n".join(history) + "\n\n"
    t = triage_ticket(context + "New message: " + message, channel, provider)
    history.append(f"Customer: {message}")
    history.append(f"Agent: {t.customer_reply}")
    return t

# Demo: two turns on the same ticket
t1 = triage_conversation("CONV-1", "My dashboard keeps timing out when I export reports.")
print("Turn 1:", t1.category, "/", t1.priority, "->", t1.customer_reply[:120], "...\n")
t2 = triage_conversation("CONV-1", "I tried a different browser and it still fails. This is now blocking my month-end report.")
print("Turn 2:", t2.category, "/", t2.priority, "escalate=", t2.escalate_to_human)
print(t2.customer_reply)

## 11. (Optional) Gradio mini-UI

A tiny interactive UI to paste a message, pick a provider, and see the triage. Optional — skip if `gradio` is not installed.

In [ ]:
try:
    import gradio as gr

    def _ui(message, provider):
        if not message.strip():
            return "Please enter a message.", ""
        t = triage_ticket(message, "chat", provider)
        internal = (
            f"category:   {t.category}\n"
            f"priority:   {t.priority}\n"
            f"department: {t.department}\n"
            f"confidence: {t.confidence:.2f}\n"
            f"escalate:   {t.escalate_to_human}  ({t.escalation_reason})\n"
            f"summary:    {t.issue_summary}"
        )
        return t.customer_reply, internal

    demo_ui = gr.Interface(
        fn=_ui,
        inputs=[gr.Textbox(lines=4, label="Customer message"),
                gr.Dropdown(["openai", "groq", "gemini"], value="openai", label="Provider")],
        outputs=[gr.Textbox(label="Customer-facing reply"),
                 gr.Textbox(label="Internal triage")],
        title="🎫 AI Support Ticket Triage",
        description="Paste a support message and see the automated triage.",
    )
    demo_ui.launch(share=False, debug=False)
except Exception as e:
    print("Gradio UI skipped:", e)

## 12. Design decisions & requirement checklist

**Why one structured call instead of many?** A single `responses.parse()` with a Pydantic schema does classification, prioritization, routing, summarizing, reply-drafting, and the escalation flag together — cheaper, lower latency, and internally consistent (the reply matches the detected priority).

**Why Structured Outputs (not "return JSON")?** They *guarantee* valid, typed fields, so the pipeline never crashes on a stray sentence or a renamed key.

**Why `temperature=0`?** Triage is a classification task — we want consistent, repeatable decisions, not creativity.

**Why a Python safety net on top of the model?** Never trust the model alone for consequential routing. Deterministic rules force human escalation on complaints, urgent/high priority, low confidence, and refund/legal/security keywords — a defensive backstop.

**Why derive department in Python?** Deterministic category → team mapping can never drift, even if the model mislabels the field.

| Requirement | ✅ Delivered by |
|-------------|----------------|
| Accept a support message | `triage_ticket()` / dataset loop / Gradio UI |
| Classify category | `category` (7 classes) |
| Detect priority | `priority` (low/medium/high/urgent) |
| Assign department | `department` + `enforce_routing()` |
| Issue summary | `issue_summary` |
| Customer-facing reply | `customer_reply` |
| Human escalation decision | `escalate_to_human` + `apply_safety_net()` |

**Deliverables produced when you run this notebook:** `support_tickets.csv` (dataset) and `triage_results.csv` (outputs), plus the printed report above. Multi-provider (OpenAI/Groq/Gemini), confidence scoring, multi-turn memory, and a Gradio UI are included as extensions.

*Possible next steps: connect to a real helpdesk/webhook, log to a database, add few-shot examples for tricky categories, and build an evaluation set to measure accuracy.*